In [1]:
!pip install -q torch transformers datasets tokenizers \
    sentence-transformers rank-bm25 pytrec-eval-terrier tqdm scipy comet-ml logging

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 96.0/96.0 kB 6.2 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 304.8/304.8 kB 23.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 787.0/787.0 kB 46.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.4/1.4 MB 69.6 MB/s eta 0:00:00


In [2]:
import os, sys, logging, json

logging.basicConfig(level=logging.INFO, format="%(asctime)s %(message)s")

In [3]:
!gdown --folder https://drive.google.com/drive/folders/1aNEW6Z-kxp8VGR-ZYS9WT0Duyx77iC2Q?usp=sharing -O /kaggle/working/

Retrieving folder contents
Processing file 1nse8vV4rja3d-l29QHQmE0glIcWFTcBO doc_ids.json
Processing file 1qPLYzXyNnjCL97lbUTVvkfCLMoJeMVxM doc_lengths.npy
Processing file 16fKNDWgKJLbLTTqH9oacxqvlzHtH8Ufw doc_offsets.npy
Processing file 1IGBSVkYfo5mvAI0jmi8Xfxe1XdxN8FTi embeddings.npy
Processing file 17TJYHfKdoo4-xgyGcxQn0BV91ZAW70Q5 metadata.json
Retrieving folder contents completed
Building directory structure
Building directory structure completed
Downloading...
From: https://drive.google.com/uc?id=1nse8vV4rja3d-l29QHQmE0glIcWFTcBO
To: /kaggle/working/index_zeroshot_colbert_xm_xmod/doc_ids.json
100%|████████████████████████████████████████| 444k/444k [00:00<00:00, 14.6MB/s]
Downloading...
From: https://drive.google.com/uc?id=1qPLYzXyNnjCL97lbUTVvkfCLMoJeMVxM
To: /kaggle/working/index_zeroshot_colbert_xm_xmod/doc_lengths.npy
100%|████████████████████████████████████████| 280k/280k [00:00<00:00, 9.35MB/s]
Downloading...
From: https://drive.google.com/uc?id=16fKNDWgKJLbLTTqH9oacxqvlzH

In [4]:
!pip install -q mteb

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 5.3/5.3 MB 65.1 MB/s eta 0:00:00


In [5]:
from diploma_code import IRMetrics, SystemMetrics, evaluate_robustness
from diploma_code import EvalConfig

2026-05-01 20:17:33,596 NumExpr defaulting to 4 threads.
2026-05-01 20:17:34,996 TensorFlow version 2.19.0 available.
2026-05-01 20:17:35,000 JAX version 0.7.2 available.
2026-05-01 20:17:43.199928: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:467] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1777666663.381621      23 cuda_dnn.cc:8579] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1777666663.443083      23 cuda_blas.cc:1407] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
W0000 00:00:1777666663.913774      23 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1777666663.913822      23 computation_placer.cc:177] computation placer already

In [ ]:
qrels = {}
with open("/kaggle/input/datasets/vikakuznetzowa/diploma-ru-hard-negatives/qrels.tsv") as f:
    for line in f:
        parts = line.strip().split("\t")
        if len(parts) >= 4:
            qid, _, did, rel = parts[:4]
            qrels.setdefault(qid, {})[did] = int(rel)

In [ ]:
from diploma_code import build_model_and_tokenizer
from diploma_code import ColBERTIndexer, ColBERTRetriever
from diploma_code import ModelConfig, IndexConfig
import torch
from huggingface_hub import hf_hub_download
import safetensors.torch

device = "cuda" if torch.cuda.is_available() else "cpu"

model_cfg = ModelConfig(encoder_name="facebook/xmod-base", embedding_dim=128)
model, tokenizer = build_model_and_tokenizer(model_cfg)

model_path = hf_hub_download("antoinelouis/colbert-xm", "model.safetensors")
hf_state_dict = safetensors.torch.load_file(model_path)

new_state_dict = {}
for k, v in hf_state_dict.items():
    if k.startswith("roberta."):
        new_k = k.replace("roberta.", "encoder.backbone.", 1)
        new_state_dict[new_k] = v
    elif k == "linear.weight":
        new_state_dict["encoder.linear.weight"] = v
    else:
        new_state_dict[k] = v

model.load_state_dict(new_state_dict, strict=False)
model.to(device)
bb = model.encoder.backbone
if hasattr(bb, 'set_default_language'):
    bb.set_default_language('ru_RU')

print("Zero-shot model loaded.")

2026-05-01 20:17:58,609 HTTP Request: HEAD https://huggingface.co/facebook/xmod-base/resolve/main/config.json "HTTP/1.1 307 Temporary Redirect"
2026-05-01 20:17:58,617 HTTP Request: HEAD https://huggingface.co/api/resolve-cache/models/facebook/xmod-base/1ff23836a9ee8b9656553630c33506a9a8a59c4f/config.json "HTTP/1.1 200 OK"
2026-05-01 20:17:58,626 HTTP Request: GET https://huggingface.co/api/resolve-cache/models/facebook/xmod-base/1ff23836a9ee8b9656553630c33506a9a8a59c4f/config.json "HTTP/1.1 200 OK"


config.json: 0.00B [00:00, ?B/s]

2026-05-01 20:17:58,667 HTTP Request: HEAD https://huggingface.co/facebook/xmod-base/resolve/main/tokenizer_config.json "HTTP/1.1 307 Temporary Redirect"
2026-05-01 20:17:58,675 HTTP Request: HEAD https://huggingface.co/api/resolve-cache/models/facebook/xmod-base/1ff23836a9ee8b9656553630c33506a9a8a59c4f/tokenizer_config.json "HTTP/1.1 200 OK"
2026-05-01 20:17:58,684 HTTP Request: GET https://huggingface.co/api/resolve-cache/models/facebook/xmod-base/1ff23836a9ee8b9656553630c33506a9a8a59c4f/tokenizer_config.json "HTTP/1.1 200 OK"


tokenizer_config.json:   0%|          | 0.00/48.0 [00:00<?, ?B/s]

2026-05-01 20:17:58,735 HTTP Request: GET https://huggingface.co/api/models/facebook/xmod-base/tree/main/additional_chat_templates?recursive=false&expand=false "HTTP/1.1 404 Not Found"
2026-05-01 20:17:58,770 HTTP Request: GET https://huggingface.co/api/models/facebook/xmod-base/tree/main?recursive=true&expand=false "HTTP/1.1 200 OK"
2026-05-01 20:17:58,796 HTTP Request: HEAD https://huggingface.co/facebook/xmod-base/resolve/main/sentencepiece.bpe.model "HTTP/1.1 404 Not Found"
2026-05-01 20:17:58,797 Warning: You are sending unauthenticated requests to the HF Hub. Please set a HF_TOKEN to enable higher rate limits and faster downloads.
2026-05-01 20:17:58,820 HTTP Request: HEAD https://huggingface.co/facebook/xmod-base/resolve/main/tokenizer.json "HTTP/1.1 307 Temporary Redirect"
2026-05-01 20:17:58,827 HTTP Request: HEAD https://huggingface.co/api/resolve-cache/models/facebook/xmod-base/1ff23836a9ee8b9656553630c33506a9a8a59c4f/tokenizer.json "HTTP/1.1 200 OK"
2026-05-01 20:17:58,835 

...

model.safetensors:   0%|          | 0.00/3.41G [00:00<?, ?B/s]

Loading weights:   0%|          | 0/4085 [00:00<?, ?it/s]

XmodModel LOAD REPORT from: facebook/xmod-base
Key                             | Status     | 
--------------------------------+------------+-
roberta.embeddings.position_ids | UNEXPECTED | 
lm_head.dense.bias              | UNEXPECTED | 
lm_head.bias                    | UNEXPECTED | 
lm_head.layer_norm.bias         | UNEXPECTED | 
lm_head.layer_norm.weight       | UNEXPECTED | 
lm_head.dense.weight            | UNEXPECTED | 
pooler.dense.weight             | MISSING    | 
pooler.dense.bias               | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.
2026-05-01 20:18:43,001 HTTP Request: HEAD https://huggingface.co/antoinelouis/colbert-xm/resolve/main/model.safetensors "HTTP/1.1 302 Found"
2026-05-01 20:18:43,126 HTTP Request: GET https://huggingface.co/api/models/antoine

model.safetensors:   0%|          | 0.00/3.41G [00:00<?, ?B/s]

Zero-shot model loaded.


In [ ]:
retriever = ColBERTRetriever(
    model, tokenizer,
    index_dir="/kaggle/working/index_zeroshot_colbert_xm_xmod",
    model_cfg=model_cfg,
    device="cuda",
)

with open("/kaggle/input/datasets/vikakuznetzowa/diploma-ru-hard-negatives/queries.jsonl") as f:
    queries_data = [json.loads(line) for line in f]
queries_dict = {q["id"]: q["text"] for q in queries_data}

eval_cfg = EvalConfig(
    robustness_typo_rate=0.2,
    robustness_num_augments=4,
    k_values=[1, 5, 10, 20, 100],
)

2026-05-01 20:19:08,666 Loaded index: 35008 docs, 4754342 vectors


In [9]:
robustness = evaluate_robustness(
    retrieve_fn=lambda q: retriever.retrieve(q, top_k=100),
    queries=queries_dict,
    qrels=qrels,
    eval_cfg=eval_cfg,
)

In [ ]:
print("\n--- Robustness degradation (%) ---")
for metric, pct in robustness["degradation_pct"].items():
    print(f"  {metric}: -{pct:.1f}%")


--- Robustness degradation (%) ---
  recall_1: -32.0%
  ndcg_cut_100: -27.6%
  map_cut_1: -32.0%
  recall_20: -25.2%
  ndcg_cut_5: -30.9%
  ndcg_cut_10: -29.9%
  ndcg_cut_20: -29.0%
  recall_5: -29.8%
  map_cut_5: -31.4%
  map_cut_100: -30.4%
  map_cut_10: -30.9%
  recip_rank: -30.4%
  recall_100: -20.2%
  map_cut_20: -30.6%
  ndcg_cut_1: -32.0%
  recall_10: -27.8%
